In [ ]:
!apt-get install -y swig cmake libgl1-mesa-dev
!pip install gymnasium[box2d] stable-baselines3 tensorflow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
The following additional packages will be installed:
  libegl-dev libgl-dev libgles-dev libgles1 libglvnd-core-dev libglvnd-dev libglx-dev libopengl-dev
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  libegl-dev libgl-dev libgl1-mesa-dev libgles-dev libgles1 libglvnd-core-dev libglvnd-dev
  libglx-dev libopengl-dev swig swig4.0
0 upgraded, 11 newly installed, 0 to remove and 30 not upgraded.
Need to get 1,336 kB of archives.
After this operation, 8,119 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglx-dev amd64 1.4.0-1 [14.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgl-dev amd64 1.4.0-1 [101 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libegl-dev amd64 1.4.

In [ ]:
#1. intialize environment
#2. initialize hyperparameters
#3. Douple NN (target , normal)
#4. Select action
#5. apply greedy of epsilon
#6. training
#7. evaluation
#8. save

In [ ]:
import tensorflow as tf
import numpy as np
import random
import gym
from collections import deque
import matplotlib.pyplot as plt
from IPython import display
import base64
import io
from PIL import Image
np.bool8 = np.bool_

# Initialize environment
env = gym.make("LunarLander-v2")

In [ ]:
#Hyperparameters and model definition
# Hyperparameters
replay_buffer_size = 200000
batch_size = 128
training_start = 256
max_episodes = 1000
max_steps = 1000
target_update_freq = 1000
train_freq = 4
discount_factor = 0.99
epsilon_start = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
early_stop_threshold = 200  # Stop if average reward reaches this value
window_size = 100  # For moving average

# Model definition
def create_model(input_shape, output_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(256, input_shape=input_shape, activation='relu'),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(output_size, activation='linear')
    ])

    @tf.function
    def train_step(model, states, targets):
        with tf.GradientTape() as tape:
            predictions = model(states, training=True)
            loss = tf.keras.losses.MSE(targets, predictions)
        gradients = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        return loss

    model.train_step = train_step
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
    return model

In [ ]:
#Helper functions and initialization
def select_action(q_values, epsilon):
    if random.random() < epsilon:
        return random.randint(0, len(q_values) - 1)
    return np.argmax(q_values)

def render_env_matplotlib(env):
    plt.figure(figsize=(8,6))
    plt.imshow(env.render(mode='rgb_array'))
    plt.axis('off')
    display.clear_output(wait=True)
    display.display(plt.gcf())
    plt.close()

# Initialize models and buffers
model = create_model((env.observation_space.shape[0] + 1,), env.action_space.n)
target_model = create_model((env.observation_space.shape[0] + 1,), env.action_space.n)
target_model.set_weights(model.get_weights())
replay_buffer = deque(maxlen=replay_buffer_size)
epsilon = epsilon_start

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Training loop with early stopping
# Training tracking
episode_rewards = []
moving_averages = []
best_average = -np.inf

for episode in range(max_episodes):
    state = env.reset()
    state = np.append(state, 0).astype(np.float32)
    episode_reward = 0

    for step in range(max_steps):
        # Get action
        state_tensor = tf.convert_to_tensor(state[np.newaxis, ...], dtype=tf.float32)
        q_values = model(state_tensor, training=False).numpy()[0]
        action = select_action(q_values, epsilon)

        # Execute action
        next_state, reward, done, _ = env.step(action)
        next_state = np.append(next_state, (step+1)/max_steps).astype(np.float32)
        episode_reward += reward

        # Store transition
        replay_buffer.append((state, action, reward, next_state, done))
        state = next_state

        # Train if enough samples
        if len(replay_buffer) >= training_start and step % train_freq == 0:
            batch = random.sample(replay_buffer, batch_size)

            # Convert to tensors
            states = tf.convert_to_tensor([t[0] for t in batch], dtype=tf.float32)
            next_states = tf.convert_to_tensor([t[3] for t in batch], dtype=tf.float32)

            # Compute targets
            next_q = target_model(next_states, training=False)
            targets = model(states, training=False).numpy()

            for i, (_, a, r, _, d) in enumerate(batch):
                if d:
                    targets[i][a] = r
                else:
                    targets[i][a] = r + discount_factor * np.max(next_q[i])

            # Train step
            model.train_step(model, states, tf.convert_to_tensor(targets, dtype=tf.float32))

        # Update target network
        if step % target_update_freq == 0:
            target_model.set_weights(model.get_weights())

        if done:
            break

    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # Track rewards
    episode_rewards.append(episode_reward)
    if len(episode_rewards) >= window_size:
        moving_avg = np.mean(episode_rewards[-window_size:])
        moving_averages.append(moving_avg)

        # Early stopping check
        if moving_avg >= early_stop_threshold:
            print(f"\nEarly stopping at episode {episode} with average reward {moving_avg:.2f}")
            break

        # Update best average
        if moving_avg > best_average:
            best_average = moving_avg
            model.save("best_model.h5")

    # Print progress
    if episode % 10 == 0:
        print(f"Episode {episode}: Reward={episode_reward:.1f}, Epsilon={epsilon:.3f}, ", end='')
        if len(moving_averages) > 0:
            print(f"Avg100={moving_averages[-1]:.1f}")
        else:
            print()

    # Save model periodically
    if episode % 100 == 0:
        model.save(f"lunar_lander_{episode}.h5")

# Save final model
model.save("lunar_lander_final.h5")

Episode 0: Reward=-377.9, Epsilon=0.995, 
Episode 10: Reward=-346.1, Epsilon=0.946, 
Episode 20: Reward=-223.0, Epsilon=0.900, 
Episode 30: Reward=-161.2, Epsilon=0.856, 
Episode 40: Reward=-93.4, Epsilon=0.814, 
Episode 50: Reward=-98.8, Epsilon=0.774, 
Episode 60: Reward=-141.3, Epsilon=0.737, 
Episode 70: Reward=-88.2, Epsilon=0.701, 
Episode 80: Reward=-142.0, Epsilon=0.666, 
Episode 90: Reward=-91.5, Epsilon=0.634, 


Episode 100: Reward=-22.9, Epsilon=0.603, Avg100=-108.7


Episode 110: Reward=-58.7, Epsilon=0.573, Avg100=-95.0


Episode 120: Reward=-53.6, Epsilon=0.545, Avg100=-88.5


Episode 130: Reward=-45.2, Epsilon=0.519, Avg100=-79.6


...

Episode 380: Reward=41.6, Epsilon=0.148, Avg100=160.3


Episode 390: Reward=-151.9, Epsilon=0.141, Avg100=166.1


Episode 400: Reward=273.5, Epsilon=0.134, Avg100=182.5


Episode 410: Reward=255.6, Epsilon=0.127, Avg100=193.2



Early stopping at episode 414 with average reward 200.70
